# Data inladen – GreatOutdoors SDM
Dit notebook laadt alle brondata (3x SQLite + 3x CSV) in het SDM.

**Cel 1** – Configuratie  
**Cel 2** – Helperfuncties  
**Cel 3** – `load()` – vul het SDM  
**Cel 4** – `clear()` – leeg het SDM  

In [6]:
# ──────────────────────────────────────────────
# CEL 1 – CONFIGURATIE
# Pas alleen hier de paden aan indien nodig
# ──────────────────────────────────────────────

import sqlite3
import pandas as pd
from pathlib import Path

SDM_PATH = "../data/sdm_deproject.db"
LOG_PATH = "../logs/source_to_sdm.log"

SQLITE_SOURCES = {
    "../GreatOutdoors/GreatOutdoors/GO_SALES-data.sqlite": [
        "order_header",
        "order_details",
        "order_method",
        "product_line",
        "product_type",
        "product",
        "sales_branch",
        "sales_staff",
        "retailer_site",
        "returned_item",
        "return_reason",
        "country",
    ],
    "../GreatOutdoors/GreatOutdoors/CRM-data.sqlite": [
        "customer",
        "customer_type",
        "customer_store",
        "customer_headquarters",
        "customer_segment",
        "customer_contact",
        "crm_country",
        "sales_territory",
        "age_group",
        "sales_demographic",
    ],
    "../GreatOutdoors/GreatOutdoors/GO_STAFF-data.sqlite": [
        "sales_representative",
        "sales_office",
        "course",
        "training",
        "satisfaction_type",
        "satisfaction",
    ],
}

CSV_SOURCES = {
    "../GreatOutdoors/GreatOutdoors/SALES_TARGET-data.csv":     "sales_target",
    "../GreatOutdoors/GreatOutdoors/PRODUCT_FORECAST-data.csv": "product_forecast",
    "../GreatOutdoors/GreatOutdoors/INVENTORY_LEVELS-data.csv": "inventory_levels",
}

print("Configuratie geladen")

Configuratie geladen


In [7]:
import logging
from datetime import datetime

def log_etl(level, message, details=""):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(LOG_PATH, "a") as f:
        f.write(f"{timestamp} | {level:10} | {message:30} | {details}\n")

def _load_from_sqlite(sdm_con, source_path, tables):
    """Kopieer meerdere tabellen uit één SQLite-bronbestand naar het SDM."""
    if not Path(source_path).exists():
        log_etl("ERROR", "Source missing", f"{source_path} not found")
        return
    src_con = sqlite3.connect(source_path)
    for table in tables:
        try:
            # 1. Controleer of tabel bestaat in SDM en haal kolommen op
            sdm_cols = [row[1] for row in sdm_con.execute(f"PRAGMA table_info({table})")]
            if not sdm_cols:
                log_etl("ERROR", f"Table {table}", "Table does not exist in SDM")
                print(f"  [SKIP]  {table:35s} (bestaat niet in SDM)")
                continue

            # 2. Lees data uit bron
            df = pd.read_sql(f"SELECT * FROM {table}", src_con)
            
            # 3. Match kolommen (case-insensitive)
            df.columns = df.columns.str.upper()
            sdm_cols_upper = [c.upper() for c in sdm_cols]
            
            common_cols = [col for col in df.columns if col in sdm_cols_upper]
            if not common_cols:
                log_etl("WARNING", f"Table {table}", "No matching columns found")
                print(f"  [SKIP]  {table:35s} (geen overeenkomende kolommen)")
                continue
                
            df = df[common_cols]
            df.columns = [sdm_cols[sdm_cols_upper.index(col)] for col in df.columns]

            # 4. Opslaan in SDM
            if not df.empty:
                df.to_sql(table, sdm_con, if_exists="append", index=False)
                log_etl("SUCCESS", f"Table {table}", f"Loaded {len(df)} records from {Path(source_path).name}")
                print(f"  [OK]   {table:35s} {len(df):>6} rijen  ←  {Path(source_path).name}")
            else:
                print(f"  [INFO] {table:35s} Geen data gevonden")
                
        except Exception as e:
            log_etl("ERROR", f"Table {table}", str(e))
            print(f"  [ERROR] {table:35s} {str(e)}")
    src_con.close()

def _load_from_csv(sdm_con, csv_path, table):
    """Laad één CSV-bestand in het SDM."""
    if not Path(csv_path).exists():
        log_etl("ERROR", "Source missing", f"{csv_path} not found")
        return

    # 1. Controleer of tabel bestaat in SDM
    sdm_cols = [row[1] for row in sdm_con.execute(f"PRAGMA table_info({table})")]
    if not sdm_cols:
        log_etl("ERROR", f"Table {table}", "Table does not exist in SDM")
        print(f"  [SKIP]  {table:35s} (bestaat niet in SDM)")
        return

    # 2. Lees CSV met encoding detectie
    for encoding in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(csv_path, encoding=encoding)
            break
        except UnicodeDecodeError:
            continue
    else:
        log_etl("ERROR", f"CSV Read {table}", "Unknown encoding")
        return
    
    try:
        # 3. Match kolommen
        df.columns = df.columns.str.upper()
        sdm_cols_upper = [c.upper() for c in sdm_cols]
        
        common_cols = [col for col in df.columns if col in sdm_cols_upper]
        if not common_cols:
            print(f"  [SKIP]  {table:35s} (geen overeenkomende kolommen)")
            return
            
        df = df[common_cols]
        df.columns = [sdm_cols[sdm_cols_upper.index(col)] for col in df.columns]
        
        df = df.drop_duplicates()
        if not df.empty:
            df.to_sql(table, sdm_con, if_exists="append", index=False)
            log_etl("SUCCESS", f"Table {table}", f"Loaded {len(df)} records from {Path(csv_path).name}")
            print(f"  [OK]   {table:35s} {len(df):>6} rijen  ←  {Path(csv_path).name}")
        else:
            print(f"  [INFO] {table:35s} Geen data gevonden")
    except Exception as e:
        log_etl("ERROR", f"Table {table}", str(e))
        print(f"  [ERROR] {table:35s} {str(e)}")

def _clear_table(sdm_con, table):
    """Verwijder alle rijen uit één SDM-tabel."""
    sdm_con.execute(f"DELETE FROM {table}")
    log_etl("INFO", f"Table {table}", "Cleared SDM table")
    print(f"  [CLEAR] {table}")

def load():
    """Vult het SDM met alle brondata uit SQLite en CSV."""
    con = sqlite3.connect(SDM_PATH)
    con.execute("PRAGMA foreign_keys = OFF;")
    print("=== LADEN ===")
    
    # 1. SQLite bronnen
    for path, tables in SQLITE_SOURCES.items():
        _load_from_sqlite(con, path, tables)
        
    # 2. CSV bronnen
    for path, table in CSV_SOURCES.items():
        _load_from_csv(con, path, table)
        
    con.execute("PRAGMA foreign_keys = ON;")
    con.close()
    print("\\n✓ Klaar – SDM gevuld.")

def clear():
    """Verwijdert alle data uit de SDM tabellen."""
    con = sqlite3.connect(SDM_PATH)
    con.execute("PRAGMA foreign_keys = OFF;")
    print("=== LEEGMAKEN ===")
    
    # Verzamel alle tabellen uit de configuratie
    all_tables = []
    for tables in SQLITE_SOURCES.values():
        all_tables.extend(tables)
    all_tables.extend(CSV_SOURCES.values())
    
    for table in all_tables:
        _clear_table(con, table)
        
    con.execute("PRAGMA foreign_keys = ON;")
    con.close()
    print("\\n✓ Klaar – SDM leeggemaakt.")


In [8]:
# ──────────────────────────────────────────────
# CEL 3 – LAAD DATA IN HET SDM
# Voer deze cel uit om het SDM te vullen
# ──────────────────────────────────────────────

load()

=== LADEN ===
  [OK]   order_header                          4968 rijen  ←  GO_SALES-data.sqlite
  [OK]   order_details                        40990 rijen  ←  GO_SALES-data.sqlite
  [OK]   order_method                             7 rijen  ←  GO_SALES-data.sqlite
  [OK]   product_line                             5 rijen  ←  GO_SALES-data.sqlite
  [OK]   product_type                            21 rijen  ←  GO_SALES-data.sqlite
  [OK]   product                                115 rijen  ←  GO_SALES-data.sqlite
  [OK]   sales_branch                            25 rijen  ←  GO_SALES-data.sqlite
  [OK]   sales_staff                             96 rijen  ←  GO_SALES-data.sqlite
  [OK]   retailer_site                          391 rijen  ←  GO_SALES-data.sqlite
  [OK]   returned_item                          690 rijen  ←  GO_SALES-data.sqlite
  [OK]   return_reason                            5 rijen  ←  GO_SALES-data.sqlite
  [OK]   country                                 18 rijen  ←  GO_SALES-da

In [5]:
# ──────────────────────────────────────────────
# CEL 4 – LEEG HET SDM
# Voer deze cel uit om alle rijen te verwijderen
# ──────────────────────────────────────────────

clear()

=== LEEGMAKEN ===


OperationalError: no such table: order_header